In [ ]:
%matplotlib widget 
%load_ext autoreload
import os
import shutil
import time
thisfiledir=os.path.abspath("")
deepracingrepodir=os.path.normpath(os.path.join(thisfiledir, "..", ".."))
import sys
sys.path = [os.path.join(deepracingrepodir, "deepracing_py"), os.path.join(deepracingrepodir, "DCNN-Pytorch"), thisfiledir] + sys.path
import matplotlib.transforms
import deepracing, deepracing_models.math_utils as mu, deepracing.path_utils
from deepracing_models.math_utils.bounds_checking import BoundsChecker
from deepracing_models.math_utils.statistics import CollisionProbabilityEstimator
from deepracing_models.probabilistic_models import ProbabilisticBezierCurve
import deepracing_models.math_utils.collision_checking as cc
import deepracing_models.data_loading.file_datasets as FD
import torch, numpy as np
from scipy.spatial.transform import Rotation, RotationSpline
import matplotlib.figure, matplotlib.axes, matplotlib.collections, matplotlib.patches, matplotlib.animation
from matplotlib import pyplot as plt
import torch.distributions
import torch.utils.data as torchdata
import scipy.interpolate, scipy.spatial
from tqdm import tqdm
import PIL, PIL.Image, PIL.ImageOps
import utils
import yaml
searchdirs = []
try:
    searchdirs.extend(os.environ["F1_MAP_DIRS"].split(os.pathsep))
except ImportError as e:
    pass
try:
    import ament_index_python # type: ignore
    searchdirs.append(os.path.join(ament_index_python.get_package_share_directory("deepracing_launch"), "maps"))
except ImportError as e:
    pass
except ament_index_python.packages.PackageNotFoundError as e:
    pass
transform_to_map=True

datadir = "/p/DeepRacing/overtaking_datasets" #/Jeddah_2023_7_13_15_27"
dsets = []
for subdir in os.listdir(datadir):
    # print(subdir)
    # with 
    with open(os.path.join(datadir, subdir,"metadata.yaml"),"r") as f:
        metadata = yaml.load(f, Loader=yaml.SafeLoader)
    with open(os.path.join(datadir, subdir,"data.npz"),"rb") as f:
        data = np.load(f)
        dsets.append(FD.OvertakingTrajectoriesDataset(data, metadata))
# delta_t = datadict["delta_t"]
concatdset = torchdata.ConcatDataset(dsets)
idx_rand = 64
# idx_rand = 138
# idx_rand = int(np.random.randint(0, high=len(concatdset), size=1))
datadict = concatdset[idx_rand]
skip=2
tfit = torch.as_tensor(datadict["delta_t"], dtype=torch.float64)[::skip]
tfit = tfit-tfit[0]
attacker_positions = torch.as_tensor(datadict["attacker_pos"]).type_as(tfit)[::skip]
defender_positions = torch.as_tensor(datadict["defender_pos"]).type_as(tfit)[::skip]
attacker_quats = torch.as_tensor(datadict["attacker_quat"]).type_as(tfit)[::skip]
defender_quats = torch.as_tensor(datadict["defender_quat"]).type_as(tfit)[::skip]

car_width = 2.0
car_length = 4.25
car_image = PIL.Image.open(os.path.join(thisfiledir, "assets", "cavalier_transparent.png"))
car_image_inverted = PIL.Image.open(os.path.join(thisfiledir, "assets", "cavalier_transparent_inverted.png"))

trackmap = deepracing.searchForTrackmap(datadict["track_name"], searchdirs, align=True, transform_to_map=transform_to_map)
Nparticles = int(round(1.25*(2**9)))
plotsdir=os.path.join(os.environ["HOME"], "plots")
videodir=os.path.join(plotsdir, "videos")
if os.path.isdir(videodir): shutil.rmtree(videodir, ignore_errors=True) 
os.makedirs(videodir, exist_ok=False)

In [ ]:

curve_covars = torch.as_tensor([[[ 1.19892773e-02,  9.36912990e-08],
                                    [ 9.36912990e-08,  1.19892773e-02]],
                                [[ 2.02386101e-01, -3.81924440e-06],
                                    [-3.81924440e-06,  2.02386101e-01]],
                                [[ 4.34330645e-01,  2.91923870e-05],
                                    [ 2.91923870e-05,  4.34330645e-01]],
                                [[ 1.14617000e+00, -9.01698636e-05],
                                    [-9.01698636e-05,  1.14617000e+00]],
                                [[ 1.94844307e+00,  1.42177334e-04],
                                    [ 1.42177334e-04,  1.94844307e+00]],
                                [[ 1.90664060e+00, -1.19909455e-04],
                                    [-1.19909455e-04,  1.90664060e+00]],
                                [[ 3.54434947e+00,  4.76160528e-05],
                                    [ 4.76160528e-05,  3.54434947e+00]],
                                [[ 1.21235046e+00, -5.08694543e-06],
                                    [-5.08694543e-06,  1.21235046e+00]]]).type_as(defender_positions)
curve_covars[:,0,1] = curve_covars[:,1,0] = 0.0
curve_covars_sparse = curve_covars.clone()
curve_covars_sparse[0,0,0]=curve_covars_sparse[0,1,1]=1E-4
curve_covars_sparse[1:]*=4.0
kbezier = curve_covars.shape[0]-1

(Mfit,), (attacker_curve,) = mu.bezierLsqfit(attacker_positions[None,...,[0,1]], kbezier, t=tfit[None])
sdense=torch.linspace(0.0, 1.0, steps=60)
tdense = sdense*(tfit[-1]-tfit[0])
Mdense, = mu.bezierM(sdense[None], kbezier).type_as(Mfit)
Mderiv, = mu.bezierM(sdense[None], kbezier-1).type_as(Mfit)
attacker_v0 = kbezier*(attacker_curve[1]-attacker_curve[0])/(tfit[-1]-tfit[0])
attacker_tau0 = attacker_v0/torch.norm(attacker_v0, p=2.0)
attacker_nu0 = attacker_tau0[[1,0]].clone()
attacker_nu0[0]*=-1.0
attacker_R0 = torch.stack([attacker_tau0, attacker_nu0], dim=1)
attacker_P0_inv = -(attacker_R0.T@attacker_curve[0,:,None])[...,0]

attacker_curve = (attacker_curve[...,None,:]@attacker_R0)[...,0,:] + attacker_P0_inv
attacker_curve_deriv = kbezier*torch.diff(attacker_curve, dim=-2)/(tfit[-1]-tfit[0])
attacker_vel = Mderiv@attacker_curve_deriv
attacker_tau = attacker_vel/torch.norm(attacker_vel, p=2.0, dim=-1, keepdim=True)
attacker_nu = attacker_tau[:,[1,0]].clone()
attacker_nu[:,0]*=-1.0
attacker_R = torch.stack([attacker_tau, attacker_nu], dim=-1)
attacker_plot = Mdense @ attacker_curve
# attacker_curve[:,0]+=3.0
# attacker_curve[:,1]-=-3.0


_, (defender_curve,) = mu.bezierLsqfit(defender_positions[None,1:-2,[0,1]], kbezier, t=tfit[None,1:-2])
defender_curve = (defender_curve[...,None,:]@attacker_R0)[...,0,:] + attacker_P0_inv
defender_curve+=torch.torch.as_tensor([-0.70, -2.5])[None].type_as(defender_curve)
defender_curve_deriv = kbezier*torch.diff(defender_curve, dim=-2)/(tfit[-1]-tfit[0])
defender_plot = Mdense @ defender_curve
defender_vel = Mderiv@defender_curve_deriv
defender_tau = defender_vel/torch.norm(defender_vel, p=2.0, dim=-1, keepdim=True)
defender_nu = defender_tau[:,[1,0]].clone()
defender_nu[:,0]*=-1.0
defender_R = torch.stack([defender_tau, defender_nu], dim=-1)


offsets = torch.linspace(0.0, 2.25*curve_covars_sparse[-1,0,0].sqrt(), steps=defender_plot.shape[0])
pTleft = defender_plot + offsets[:,None]*defender_nu
pTright = defender_plot - offsets[:,None]*defender_nu
pTpolypoints = torch.cat([pTleft, pTright.flip(0)[:-1]], dim=0)

targetcurve_dist = torch.distributions.MultivariateNormal(defender_curve, covariance_matrix=curve_covars_sparse)
sampled_curves = targetcurve_dist.sample([2**12,])
sampled_curve_derivs = kbezier*torch.diff(sampled_curves, dim=-2)/(tfit[-1]-tfit[0])
sampled_curve_points = Mdense @ sampled_curves
sampled_curve_vels = Mderiv @ sampled_curve_derivs
sampled_curve_tau = sampled_curve_vels/torch.norm(sampled_curve_vels, p=2.0, dim=-1, keepdim=True)
sampled_curve_nu = sampled_curve_tau[...,[1,0]].clone()   
sampled_curve_nu[...,0]*=-1.0
sampled_curve_R = torch.stack([sampled_curve_tau, sampled_curve_nu], dim=-1)



image_scale=2.0
boxpoints01 = (torch.stack(torch.meshgrid([
    0.5*torch.as_tensor([-car_length, car_length]),
    0.5*torch.as_tensor([-car_width, car_width])
], indexing="ij"), dim=0).reshape(2,-1).type_as(attacker_plot))
boxpoints01 = image_scale*boxpoints01[:,torch.argsort(torch.atan2(boxpoints01[1], boxpoints01[0]))]

boxpoints_attacker = (attacker_R @ boxpoints01).transpose(-2,-1) + attacker_plot[...,None,:]
boxpoints_defender = (defender_R @ boxpoints01).transpose(-2,-1) + defender_plot[...,None,:]
boxpoints_samples = (sampled_curve_R @ boxpoints01).transpose(-2,-1) + sampled_curve_points[...,None,:]
collision_result_dense = cc.rectangle_intersections2(boxpoints_attacker[None].expand_as(boxpoints_samples), boxpoints_samples, check_singular=True)

figsize=8.0*np.ones(2, dtype=np.float64)
figname="sampled curves"
plt.close(fig=figname)
fig, ax = plt.subplots(figsize=figsize, frameon=False, label=figname, layout="compressed")
ax.set_axis_off()
ax.set_aspect(1.0, adjustable='box')
attacker_line, = ax.plot(*(attacker_plot.T.cpu()), label="$\\mathcal{T}_{ego}$", color=utils.COLORS.UVA_ORANGE)
fig.savefig(os.path.join(plotsdir, "only_Tego.svg"), transparent=True, pad_inches=0, bbox_inches="tight")
patch : matplotlib.patches.Polygon = ax.add_patch(matplotlib.patches.Polygon(pTpolypoints.cpu(), closed=True, facecolor=1.0-utils.COLORS.UVA_ORANGE, edgecolor=None))
patch.set_alpha(.5)
fig.savefig(os.path.join(plotsdir,  "Tego_and_pT.svg"), facecolor='none', transparent=True, pad_inches=0.0, bbox_inches="tight")
patch.set_visible(True)
attacker_line.set_visible(False)
fig.savefig(os.path.join(plotsdir, "only_pT.svg"), transparent=True, pad_inches=0, bbox_inches="tight")
patch.set_visible(False)
attacker_line.set_visible(True)
fig.savefig(os.path.join(plotsdir, "only_Tego.svg"), transparent=True, pad_inches=0, bbox_inches="tight")
# patch.remove()
# ax.plot(*(defender_plot.T.cpu()), label="$p(\\mathcal{T})$", color=1.0-utils.COLORS.UVA_ORANGE, linestyle="--")
static_artists = [attacker_line,patch]
sample_frames = 16
artists = [static_artists,]
for j in range(sample_frames):
    current_artists = ax.plot(*(sampled_curve_points[j].T.cpu()), color=utils.COLORS.UVA_BLUE, alpha=1.0, linestyle="--")
    artists.append(static_artists + current_artists)
total_animation_time=5.0
fps = sample_frames/total_animation_time
#writer="ffmpeg",
animation = matplotlib.animation.ArtistAnimation(fig, artists, interval=int(round(1000.0/fps)), repeat=False)
animation.save(os.path.join(videodir, "overtaking_sampled_curves.mp4"),dpi=400, writer="ffmpeg",  fps=fps, savefig_kwargs={'facecolor':'none', "transparent" : True, "pad_inches" : 0.0})
plt.close(fig=fig)



In [ ]:
figname="montecarlo video"
plt.close(fig=figname)
fig, ax = plt.subplots(figsize=figsize, label=figname)
static_artists = ax.plot(*(attacker_plot.T.cpu()), label="$\\mathcal{T}_{ego}$", color=utils.COLORS.UVA_ORANGE)
ax.set_axis_off()
ax.set_aspect(1.0, adjustable='box')
artists = []
num_samples = 8
framedir=os.path.join(videodir, "montecarlo")
if os.path.isdir(framedir): shutil.rmtree(framedir, ignore_errors=True)
os.makedirs(framedir, exist_ok=False)
for sample_idx in range(num_samples):
# static_artists.extend(ax.plot(*(sampled_curve_points[sample_idx].T.cpu()), color=utils.COLORS.UVA_BLUE, linestyle="--"))
    current_artists = ax.plot(*(sampled_curve_points[sample_idx].T.cpu()), color=1.0-utils.COLORS.UVA_ORANGE, linestyle="--")
    idx_col = collision_result_dense.collision_idx[sample_idx]
    for i in range(Mdense.shape[0]):
        current_artists.append(ax.add_patch(matplotlib.patches.Polygon(boxpoints_attacker[i].cpu(), closed=True, fill=False, edgecolor=static_artists[0].get_color(), alpha=1.0)))
        color="red" if idx_col[i] else current_artists[0].get_color()
        current_artists.append(ax.add_patch(matplotlib.patches.Polygon(boxpoints_samples[sample_idx,i].cpu(), closed=True, fill=False, edgecolor=color, alpha=1.0)))
        artists.append(current_artists + static_artists)
    
# total_animation_time=0.75*num_samples
fps = 30.0
animation = matplotlib.animation.ArtistAnimation(fig, artists, interval=int(round(1000.0/fps)), repeat=True)
animation.save(os.path.join(videodir, "montecarlo.mp4"), writer="ffmpeg", fps=fps)
plt.close(fig=fig)

In [ ]:
figname="montecarlo"
plt.close(fig=figname)
fig, ax = plt.subplots(figsize=figsize, label=figname)
ax.set_axis_off()
ax.set_aspect(1.0, adjustable='box')
ax.plot(*(attacker_plot.T.cpu()), label="$\\mathcal{T}_{ego}$", color=utils.COLORS.UVA_ORANGE)
idx_any_collision = (collision_result_dense.collision_idx.sum(dim=1)>0)
sampleplots=[]
for sample_idx in range(num_samples):
    sampleplots.extend(ax.plot(*(sampled_curve_points[sample_idx].T.cpu()), color=1.0-utils.COLORS.UVA_ORANGE, linestyle="--"))
fig.savefig(os.path.join(plotsdir, "videos", "montecarlo", "allsamples.svg"), transparent=True, pad_inches=0, bbox_inches="tight")
for sample_idx in range(num_samples):
    if idx_any_collision[sample_idx]:
        sampleplots[sample_idx].set_color("red")
fig.savefig(os.path.join(plotsdir, "videos", "montecarlo", "allsamples_colored.svg"), transparent=True, pad_inches=0, bbox_inches="tight")
    


In [ ]:

from seaborn import axes_style
figsize=8.0*np.ones(2, dtype=np.float64)
plt.close("all")
fig, ax = plt.subplots(figsize=figsize,frameon=False)
axtime = fig.add_axes([0.05, 0.5, 0.55, 0.45])
ax.set_axis_off()
ax.set_aspect(1.0, adjustable='box')


allpoints = torch.cat([defender_plot, attacker_plot], dim=0)

xmin = allpoints[:,0].min().item() - car_length
xmax = allpoints[:,0].max().item() + car_length
ymin = allpoints[:,1].min().item() - car_length
ymax = allpoints[:,1].max().item() + car_length
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

static_artists = ax.plot(*(attacker_plot.T.cpu()), label="$\\mathcal{T}_{ego}$", color=utils.COLORS.UVA_ORANGE)
# static_artists.extend(ax.plot(*(defender_plot.T.cpu()), label="$p(\\mathcal{T})$", color=1.0-utils.COLORS.UVA_ORANGE, linestyle="--"))
static_artists.append(ax.add_patch(matplotlib.patches.Polygon(pTpolypoints.cpu(), closed=True, facecolor=1.0-utils.COLORS.UVA_ORANGE, edgecolor=None, alpha=0.5)))

collision_result = cc.rectangle_intersections2(boxpoints_attacker, boxpoints_defender)
idx_collision = collision_result.collision_idx
gt_collision_probs = collision_result_dense.collision_idx.sum(dim=0)/collision_result_dense.collision_idx.shape[0]
alpha=0.35
gt_lambda = alpha*gt_collision_probs + (1-alpha)*(gt_collision_probs/(1-gt_collision_probs))
print(gt_lambda)

car_diameter=image_scale*torch.as_tensor([car_length, car_width], dtype=torch.float64).norm(p=2.0).item()

deltas = sampled_curve_points - attacker_plot[None]
delta_norms = torch.norm(deltas, p=2.0, dim=-1)

idx_collision_circle = delta_norms<(0.65*car_diameter)
circle_collision_probs = idx_collision_circle.sum(dim=0)/idx_collision_circle.shape[0]

sample_frames = int(attacker_plot.shape[0])
total_animation_time=6.0
fps = sample_frames/total_animation_time
artists = []
Ncol=0
withtext=False
# outsize=(3000,3000)
# fourcc = cv2.VideoWriter.fourcc(*'MJPG')
# video_writer = cv2.VideoWriter(os.path.join(videodir, "overtake_animation.avi"), fourcc, fps, outsize)
axtime.set_xlim(0.0, tdense[-1]+0.1)
ncolarr = [0,]
axtime.yaxis.tick_right()
# axtime.set_ylim(0.0, gt_lambda.max().item()+.2)
axtime.set_ylim(0.0, 1.0)
# axtime.yaxis.set_ticks(np.arange(0, 21, step=4, dtype=np.int64))

for j in range(sample_frames):
    _, _, imartist_defender = utils.plot_image(ax, car_image_inverted, defender_plot[j].cpu(), defender_tau[j].cpu(), car_width, car_length, image_scale=image_scale)
    _, _, imartist_attacker = utils.plot_image(ax, car_image, attacker_plot[j].cpu(), attacker_tau[j].cpu(), car_width, car_length, image_scale=image_scale)

    # edgecolor = "red" if idx_collision[j] else "green"
    Ncol+=int(idx_collision[j].cpu())
    # print(Ncol)
    # box_attacker = ax.add_patch(matplotlib.patches.Polygon(boxpoints_attacker[j].cpu(), closed=True, fill=False, edgecolor=edgecolor))
    # box_defender = ax.add_patch(matplotlib.patches.Polygon(boxpoints_defender[j].cpu(), closed=True, fill=False, edgecolor=edgecolor))
    current_artists = [imartist_defender, imartist_attacker]#, box_attacker, box_defender,]
    # if withtext:
    #     current_artists.append(ax.text(0.5, 0.5, "$N(t)=%d$" % (Ncol,), transform=fig.transFigure, fontsize=15, color="black"))
    # if j<(sample_frames-1):
    # current_artists.append(axtime.scatter(tdense[:j+1], ncolarr, color="black", s=2**1.5))
    if j>0:
        # current_artists.extend(axtime.plot(tdense[:j], gt_collision_probs[:j], color="black", linestyle="--", linewidth=1.25, label=r"$P_{col}(t)$"))
        # current_artists.extend(axtime.plot(tdense[:j], gt_lambda[:j], color="black",  linewidth=1.25, label=r"$\lambda(t)$"))
        current_artists.extend(axtime.plot(tdense[:j], circle_collision_probs[:j], color="black", linewidth=1.25))
    # elif j==1:
    #     current_artists.append(axtime.legend(frameon=False, loc="upper left", fontsize=15))
    ncolarr.append(Ncol)
    artists.append(static_artists + current_artists)
    

    
    # buf = io.BytesIO()
    # fig.savefig(buf, format='png', dpi=300, transparent=True, pad_inches=0, bbox_inches="tight")
    # buf.seek(0)
    # pil_img = PIL.Image.open(buf).resize(outsize)
    # frame = np.array(pil_img)
    # video_writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))


    # imartist_defender.remove()
    # imartist_attacker.remove()
    # box_attacker.remove()
    # box_defender.remove()  
# video_writer.release() 
animation = matplotlib.animation.ArtistAnimation(fig, artists, interval=int(round(1000.0/fps)), repeat=True, blit=True)
animation.save(os.path.join(videodir, "overtake_animation%s.mp4" % ("_withtext" if withtext else "",)), writer="ffmpeg",  fps=fps, dpi=300, savefig_kwargs={"transparent": True, "pad_inches": 0})

for frame in artists[:-1]:
    # print(frame)
    for artist in frame:
        try:
            artist.remove()
        except Exception as e:
            pass
axtime.fill_between(tdense.cpu(), torch.zeros_like(gt_lambda).cpu(), gt_lambda.cpu(), color="black", alpha=0.25)
fig.savefig(os.path.join(plotsdir, "videos", "overtake_animation_final.svg"), transparent=True, pad_inches=0, bbox_inches="tight")
plt.close(fig=fig)

In [ ]:
from scipy.spatial.transform import Rotation
#image_scale*
#image_scale*
lat_buffer = 0.5*image_scale*car_width
long_buffer = 0.5*image_scale*car_length
cpe : CollisionProbabilityEstimator = CollisionProbabilityEstimator(8, tfit[-1].item(), 8, lat_buffer, long_buffer).requires_grad_(False)

Mgl1d, = mu.bezierM((cpe.gl1d.eta.detach().clone())[None]/tfit[-1], kbezier)
Mgl1d_deriv, = mu.bezierM((cpe.gl1d.eta.detach().clone())[None]/tfit[-1], kbezier-1)
flip=torch.as_tensor([-1.0, 1.0]).type_as(Mgl1d)

attacker_pos_gl1d = Mgl1d @ attacker_curve
attacker_vel_gl1d = Mgl1d_deriv @ attacker_curve_deriv
attacker_tau_gl1d = attacker_vel_gl1d/torch.norm(attacker_vel_gl1d, p=2.0, dim=-1, keepdim=True)
attacker_nu_gl1d = attacker_tau_gl1d[:,[1,0]].clone()*flip[None]
attacker_R_gl1d = torch.stack([attacker_tau_gl1d, attacker_nu_gl1d], dim=-1)

defender_pos_gl1d = Mgl1d @ defender_curve
defender_vel_gl1d = Mgl1d_deriv @ defender_curve_deriv
defender_tau_gl1d = defender_vel_gl1d/torch.norm(defender_vel_gl1d, p=2.0, dim=-1, keepdim=True)
defender_nu_gl1d = defender_tau_gl1d[:,[1,0]].clone()*flip[None]
defender_R_gl1d = torch.stack([defender_tau_gl1d, defender_nu_gl1d], dim=-1)

fig_gl1d, ax_gl1d = plt.subplots(figsize=figsize, frameon=False)

ax_gl1d.set_aspect(1.0, adjustable='box')
ax_gl1d.set_axis_off()
ax_gl1d.plot(*(attacker_plot.T.cpu()), label="$\\mathcal{T}_{ego}$", color=utils.COLORS.UVA_ORANGE, zorder=1)
ax_gl1d.plot(*(defender_plot.T.cpu()), label="$p(\\mathcal{T})$", color=1.0-utils.COLORS.UVA_ORANGE, zorder=1)
ax_gl1d.set_xlim(xmin, xmax)
ax_gl1d.set_ylim(ymin, ymax)
fig_gl1d.savefig(os.path.join(plotsdir, "overtake_blank.svg"), bbox_inches="tight", pad_inches=0.0)
gausspoints01 = cpe.gaussian_pdf_integrator.eta_01.detach().clone().T
# print(gausspoints01)
# print(gausspoints01.shape)
for idx in range(attacker_pos_gl1d.shape[0]):
    _, _, imshowartist1 = utils.plot_image(ax_gl1d, car_image, attacker_pos_gl1d[idx].cpu(), attacker_tau_gl1d[idx].cpu(), car_width, car_length, image_scale=image_scale, zorder=2)
    _, _, imshowartist2 = utils.plot_image(ax_gl1d, car_image_inverted, defender_pos_gl1d[idx].cpu(), defender_tau_gl1d[idx].cpu(), car_width, car_length, image_scale=image_scale, zorder=2)
    boxpoints_defender_gl1d = (defender_R_gl1d[idx]@boxpoints01).transpose(-2,-1) + defender_pos_gl1d[idx,None]
    boxpoints_attacker_gl1d = (attacker_R_gl1d[idx]@boxpoints01).transpose(-2,-1) + attacker_pos_gl1d[idx,None]
    gl2d_points = (attacker_R_gl1d[idx]@gausspoints01).transpose(-2,-1) + attacker_pos_gl1d[idx,None]
    # ax_gl1d.add_patch(matplotlib.patches.Polygon(boxpoints.cpu(), closed=True, fill=False, edgecolor=utils.COLORS.UVA_ORANGE))
    # ax_gl1d.scatter(*(gl2d_points.cpu().T), color=utils.COLORS.UVA_ORANGE, zorder=3, s=1.5**2)
    subfig, subax = plt.subplots(figsize=figsize, frameon=False)
    subax.set_axis_off()
    subax.set_aspect(1.0, adjustable='box')
    subax.add_patch(matplotlib.patches.Polygon(boxpoints_defender_gl1d.cpu(), closed=True, fill=False, edgecolor=1.0-utils.COLORS.UVA_ORANGE))
    subax.add_patch(matplotlib.patches.Polygon(boxpoints_attacker_gl1d.cpu(), closed=True, fill=False, edgecolor=utils.COLORS.UVA_ORANGE))
    subax.scatter(*(gl2d_points.cpu().T), color=utils.COLORS.UVA_ORANGE, zorder=3, s=1.5**2)
    subfig.savefig(os.path.join(plotsdir, "overtake_gl1d_box{:d}.svg".format(idx)), bbox_inches="tight", pad_inches=0.0)
    plt.close(fig=subfig)
fig_gl1d.savefig(os.path.join(plotsdir, "overtake_gl1d.svg"), bbox_inches="tight", pad_inches=0.0)

subfig, subax = plt.subplots(figsize=1.0*np.asarray([.98,.46]), frameon=False)
subax.set_axis_off()
subax.set_aspect(1.0, adjustable='box')
subax.scatter(*(gausspoints01.cpu()), color=utils.COLORS.UVA_ORANGE, zorder=3, s=2.0**.25)
subfig.savefig(os.path.join(plotsdir, "gl1d_01.svg".format(idx)), bbox_inches="tight", pad_inches=0.0)
plt.close(fig=subfig)

mean, stdevs = torch.zeros_like(attacker_plot[0]), 1.5*torch.ones_like(attacker_plot[0])
angle = 0.0
R = torch.as_tensor(Rotation.from_rotvec([0.0,0.0,angle]).as_matrix()[0:2,0:2]).type_as(mean)
covariance = R@torch.diag_embed(torch.square(stdevs))@R.T
normal_distribution = torch.distributions.MultivariateNormal(mean, covariance_matrix=covariance)

pdf0 : torch.Tensor = normal_distribution.log_prob(mean).exp()


ri = torch.linspace(0.0, car_width/2.0, steps=200).type_as(mean)
thetai = torch.linspace(-np.pi, np.pi, steps=ri.shape[0]).type_as(mean)
Ri, Thetai = torch.meshgrid(ri, thetai, indexing="ij")
Xi = Ri*torch.cos(Thetai) + mean[0]
Yi = Ri*torch.sin(Thetai) + mean[1]
Pointsi = torch.stack([Xi,Yi], dim=-1)
#/pdf0
Pdfi = (normal_distribution.log_prob(Pointsi).exp())**1.3
fig_gaussian, ax_gaussian = plt.subplots(figsize=0.25*figsize, frameon=False, subplot_kw={'projection': 'polar'})
ax_gaussian.set_axis_off()
ax_gaussian.set_aspect(1.0, adjustable='box')
# ax_gaussian.set_xlim(mean[0]-stdevs[0], mean[0]+stdevs[0])
# ax_gaussian.set_ylim(mean[1]-stdevs[1], mean[1]+stdevs[1])
# cntr1 = ax_gaussian.contourf(Xi, Yi, Pdfi, levels=14, cmap="jet")
cntr1 = ax_gaussian.contourf(Thetai, Ri, Pdfi, levels=ri.shape[0], cmap="jet")
# utils.plot_gaussian(ax_gaussian, mean, stdevs, angle, num_ellipses=100)
fig_gaussian.savefig(os.path.join(plotsdir, "gaussian.svg"), bbox_inches="tight", pad_inches=0.0)
# plt.close(fig=fig_gaussian)